In [ ]:
!ls ''


### Khám phá nội dung của thư mục dữ liệu (EDA Bước 1)
Đầu tiên, hãy liệt kê các file bên trong thư mục `metadata` để tìm file CSV và xem trước một vài file trong thư mục `audio_data`.

In [ ]:
import os

base_dir = ''
audio_dir = os.path.join(base_dir, 'audio_data')
metadata_dir = os.path.join(base_dir, 'metadata')

# Kiểm tra các file trong thư mục metadata
print("--- Files in metadata ---")
metadata_files = os.listdir(metadata_dir)
for file in metadata_files:
    print(file)

# Kiểm tra một số file trong thư mục audio_data
print("\n--- First 5 files in audio_data ---")
audio_files = os.listdir(audio_dir)
for file in audio_files[:5]:
    print(file)

print(f"\nTổng số lượng file audio: {len(audio_files)}")

--- Files in metadata ---
train_phones.csv
Readme.md
.DS_Store
lexicon_vmd.txt
train.csv

--- First 5 files in audio_data ---
.DS_Store
train

Tổng số lượng file audio: 2


### EDA Bước 2: Phân tích file CSV và thư mục con chứa Audio
Đọc dữ liệu bảng từ `train.csv` và kiểm tra các file bên trong thư mục `audio_data/train`.

In [ ]:
import pandas as pd

# 1. Đọc và khám phá file train.csv
train_csv_path = os.path.join(metadata_dir, 'train.csv')
df_train = pd.read_csv(train_csv_path)

print("--- Kích thước dữ liệu train.csv ---")
print(df_train.shape)

print("\n--- 5 dòng đầu tiên của train.csv ---")
display(df_train.head())

# 2. Khám phá thư mục con 'train' bên trong 'audio_data'
train_audio_dir = os.path.join(audio_dir, 'train')
if os.path.isdir(train_audio_dir):
    actual_audio_files = [f for f in os.listdir(train_audio_dir) if not f.startswith('.')]
    print(f"\n--- Tổng số file audio thực tế trong thư mục 'train': {len(actual_audio_files)} ---")
    print("5 file đầu tiên:")
    for f in actual_audio_files[:5]:
        print(f)
else:
    print(f"\nKhông tìm thấy thư mục {train_audio_dir}")

### EDA Bước 3: Trực quan hóa âm thanh
Nghe thử và vẽ Waveform (dạng sóng) cùng Spectrogram (phổ đồ) của file âm thanh đầu tiên trong tập huấn luyện.

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd

# Lấy đường dẫn của file âm thanh đầu tiên từ dataframe
first_audio_relative_path = df_train['path'].iloc[0]
audio_path = os.path.join(base_dir, first_audio_relative_path)

print(f"File đang phân tích: {audio_path}")
print(f"Transcript thực tế: {df_train['transcript'].iloc[0]}")

# Tải file âm thanh bằng librosa
# sr=None giữ nguyên tần số lấy mẫu (sample rate) gốc của file
y, sr = librosa.load(audio_path, sr=None)
print(f"Sample rate (tần số lấy mẫu): {sr} Hz")
print(f"Độ dài (samples): {len(y)}")
print(f"Thời lượng: {len(y)/sr:.2f} giây")

# Hiển thị trình phát âm thanh ngay trên Colab
display(ipd.Audio(y, rate=sr))

# Cài đặt kích thước cho biểu đồ
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(14, 10))

# 1. Vẽ Waveform (Dạng sóng)
librosa.display.waveshow(y, sr=sr, ax=ax[0])
ax[0].set(title='Waveform (Biên độ theo thời gian)', xlabel='Thời gian (s)', ylabel='Biên độ')

# 2. Vẽ Spectrogram (Phổ đồ)
# Tính STFT (Short-Time Fourier Transform)
D = librosa.stft(y)
# Chuyển đổi biên độ sang decibel (dB)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
img = librosa.display.specshow(S_db, x_axis='time', y_axis='hz', sr=sr, ax=ax[1])
fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
ax[1].set(title='Spectrogram (Tần số theo thời gian)', xlabel='Thời gian (s)', ylabel='Tần số (Hz)')

plt.tight_layout()
plt.show()

### EDA Bước 4: Phân tích thống kê thời lượng âm thanh
Tính toán độ dài (giây) của tất cả file âm thanh, vẽ biểu đồ phân bố và tìm ra các file có thời lượng đặc biệt.

In [ ]:
import librosa
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

base_dir = ''
metadata_dir = os.path.join(base_dir, 'metadata')

# Đảm bảo df_train được tải nếu chưa có trong phiên làm việc
if 'df_train' not in locals() and 'df_train' not in globals():
    train_csv_path = os.path.join(metadata_dir, 'train.csv')
    df_train = pd.read_csv(train_csv_path)

# Khởi tạo list để lưu thời lượng
durations = []

# Quét qua tất cả các file trong dataframe để lấy độ dài
print("Đang tính toán thời lượng cho tất cả các file âm thanh...")
for relative_path in tqdm(df_train['path']):
    full_path = os.path.join(base_dir, relative_path)
    # Lấy thời lượng (sử dụng librosa.get_duration sẽ nhanh hơn load toàn bộ file)
    duration = librosa.get_duration(path=full_path)
    durations.append(duration)

# Thêm cột duration vào dataframe
df_train['duration'] = durations

# Tính toán các thông số thống kê
min_duration = df_train['duration'].min()
max_duration = df_train['duration'].max()
mean_duration = df_train['duration'].mean()

print(f"\n--- Thống kê Thời lượng ---")
print(f"Tổng số file: {len(df_train)}")
print(f"Ngắn nhất: {min_duration:.2f} giây")
print(f"Dài nhất: {max_duration:.2f} giây")
print(f"Trung bình: {mean_duration:.2f} giây")

# Vẽ biểu đồ phân bố (Histogram)
plt.figure(figsize=(10, 5))
sns.histplot(df_train['duration'], bins=50, kde=True, color='skyblue')
plt.title('Phân bố Thời lượng của các File Âm thanh')
plt.xlabel('Thời lượng (giây)')
plt.ylabel('Số lượng file')
plt.grid(axis='y', alpha=0.75)
plt.show()

# In thông tin chi tiết về file ngắn nhất và dài nhất
shortest_file = df_train.loc[df_train['duration'].idxmin()]
longest_file = df_train.loc[df_train['duration'].idxmax()]

print("\n--- Chi tiết File ngắn nhất ---")
print(shortest_file[['path', 'duration', 'transcript']])

print("\n--- Chi tiết File dài nhất ---")
print(longest_file[['path', 'duration', 'transcript']])

### EDA Bước 5: Trích xuất và trực quan hóa các đặc trưng âm thanh nâng cao

Chúng ta sẽ trích xuất và hiển thị hai loại đặc trưng phổ biến:

1.  **Mel-frequency Cepstral Coefficients (MFCCs)**: Được sử dụng rộng rãi trong nhận dạng giọng nói và âm nhạc, MFCCs mô tả hình dạng bao năng lượng của phổ tần số, phản ánh timbre (âm sắc) của âm thanh.
2.  **Chroma Features**: Đại diện cho phân bố năng lượng theo cao độ (pitch class) trong một đoạn âm thanh, hữu ích cho việc phân tích hòa âm và giai điệu.

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Sử dụng lại y và sr từ bước trước (EDA Bước 3)
# Nếu bạn chạy lại notebook từ đây, hãy đảm bảo các biến y, sr, audio_path đã được định nghĩa

print(f"Đang trích xuất và trực quan hóa MFCCs và Chroma features cho: {audio_path}")

# 1. Trích xuất MFCCs
# n_mfcc là số lượng hệ số MFCC muốn trích xuất
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

# 2. Trích xuất Chroma Features
# Để có chroma features tốt, thường cần hạ tần số lấy mẫu (resample) hoặc sử dụng harmonizer
chroma = librosa.feature.chroma_stft(y=y, sr=sr)

# Cài đặt kích thước cho biểu đồ
fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(14, 15), sharex=True)

# 1. Vẽ Waveform (lặp lại để có cái nhìn tổng quan)
librosa.display.waveshow(y, sr=sr, ax=ax[0])
ax[0].set(title='Waveform (Biên độ theo thời gian)', ylabel='Biên độ')

# 2. Vẽ MFCCs
img_mfcc = librosa.display.specshow(mfccs, x_axis='time', ax=ax[1])
fig.colorbar(img_mfcc, ax=[ax[1]])
ax[1].set(title='MFCCs (Mel-frequency Cepstral Coefficients)', ylabel='Hệ số MFCC')

# 3. Vẽ Chroma Features
img_chroma = librosa.display.specshow(chroma, y_axis='chroma', x_axis='time', ax=ax[2])
fig.colorbar(img_chroma, ax=[ax[2]])
ax[2].set(title='Chroma Features', ylabel='Pitch Class')

plt.tight_layout()
plt.show()

### EDA Bước 6: Khám phá thêm Dữ liệu CSV

Bây giờ chúng ta sẽ đi sâu hơn vào `df_train` để hiểu rõ cấu trúc, kiểu dữ liệu và kiểm tra các giá trị thiếu (missing values) hoặc trùng lặp.

In [ ]:
# 1. Hiển thị thông tin tổng quan về DataFrame (cột, kiểu dữ liệu, số lượng non-null)
print("--- Thông tin tổng quan về df_train ---")
df_train.info()

# 2. Kiểm tra số lượng giá trị thiếu (missing values) mỗi cột
print("\n--- Kiểm tra giá trị thiếu (Missing Values) ---")
print(df_train.isnull().sum())

# 3. Kiểm tra số lượng bản ghi trùng lặp
print("\n--- Kiểm tra bản ghi trùng lặp ---")
duplicated_rows = df_train.duplicated().sum()
print(f"Số lượng bản ghi trùng lặp: {duplicated_rows}")

# 4. Hiển thị các giá trị duy nhất và tần suất cho các cột có thể phân loại (categorical) hoặc có ít giá trị duy nhất
# Ví dụ: kiểm tra cột 'id' để xem có trùng lặp không (mỗi ID nên là duy nhất cho mỗi file audio)
print("\n--- Kiểm tra tính duy nhất của cột 'id' ---")
print(f"Số lượng ID duy nhất: {df_train['id'].nunique()} / {len(df_train)}")


--- Thông tin tổng quan về df_train ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3180 entries, 0 to 3179
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          3180 non-null   object 
 1   path        3180 non-null   object 
 2   canonical   3180 non-null   object 
 3   transcript  3180 non-null   object 
 4   duration    3180 non-null   float64
dtypes: float64(1), object(4)
memory usage: 124.3+ KB

--- Kiểm tra giá trị thiếu (Missing Values) ---
id            0
path          0
canonical     0
transcript    0
duration      0
dtype: int64

--- Kiểm tra bản ghi trùng lặp ---
Số lượng bản ghi trùng lặp: 0

--- Kiểm tra tính duy nhất của cột 'id' ---
Số lượng ID duy nhất: 3180 / 3180


In [ ]:
import editdistance
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# --- FIX: Đảm bảo df_mdd được định nghĩa bằng cách load và merge dữ liệu ---
base_dir = ''
metadata_dir = os.path.join(base_dir, 'metadata')

train_csv_path = os.path.join(metadata_dir, 'train.csv')
train_phones_csv_path = os.path.join(metadata_dir, 'train_phones.csv')

# Đọc dữ liệu
df_base = pd.read_csv(train_csv_path)
df_p = pd.read_csv(train_phones_csv_path)

# Merge để lấy canonical và transcript từ file phones
df_mdd = pd.merge(
    df_base[['id', 'path']],
    df_p[['id', 'canonical', 'transcript']],
    on='id',
    how='inner'
)

# Đổi tên cột để khớp với logic tính toán bên dưới
df_mdd.rename(columns={'canonical': 'target_phones', 'transcript': 'actual_phones'}, inplace=True)

# 1. Tính toán độ chênh lệch
def calculate_diff_percentage(row):
    can = str(row['target_phones']).split()
    trans = str(row['actual_phones']).split()
    dist = editdistance.eval(can, trans)
    return (dist / len(can)) * 100 if len(can) > 0 else 0

df_mdd['diff_percentage'] = df_mdd.apply(calculate_diff_percentage, axis=1)

# 2. Thống kê
mean_diff = df_mdd['diff_percentage'].mean()
perfect_match = (df_mdd['diff_percentage'] == 0).sum()

print(f"--- Thống kê độ chênh lệch giữa Canonical và Actual Transcript ---")
print(f"Tỉ lệ chênh lệch trung bình: {mean_diff:.2f}%")
print(f"Số lượng mẫu khớp hoàn toàn (0% diff): {perfect_match} / {len(df_mdd)} ({perfect_match/len(df_mdd)*100:.2f}%)")

# Hiển thị mẫu
print("\n--- Top 5 mẫu có độ chênh lệch cao nhất ---")
display(df_mdd.sort_values(by='diff_percentage', ascending=False).head(5))

### EDA Bước 7: Phân tích cột `transcript`

Cột `transcript` chứa nội dung văn bản của mỗi file âm thanh. Chúng ta sẽ phân tích cột này để hiểu về độ dài của các transcript và kiểm tra các giá trị duy nhất (nếu có).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm cột độ dài transcript
df_train['transcript_length'] = df_train['transcript'].apply(len)

print("--- Thống kê độ dài Transcript ---")
print(df_train['transcript_length'].describe())

# Vẽ biểu đồ phân bố độ dài transcript
plt.figure(figsize=(10, 5))
sns.histplot(df_train['transcript_length'], bins=50, kde=True, color='purple')
plt.title('Phân bố Độ dài Transcript')
plt.xlabel('Độ dài Transcript (số ký tự)')
plt.ylabel('Số lượng Transcript')
plt.grid(axis='y', alpha=0.75)
plt.show()

# Hiển thị một số transcript ngắn nhất và dài nhất
print("\n--- 5 Transcript ngắn nhất ---")
shortest_transcripts = df_train.sort_values(by='transcript_length').head(5)
for index, row in shortest_transcripts.iterrows():
    print(f"Độ dài: {row['transcript_length']}, Transcript: '{row['transcript']}'")

print("\n--- 5 Transcript dài nhất ---")
longest_transcripts = df_train.sort_values(by='transcript_length', ascending=False).head(5)
for index, row in longest_transcripts.iterrows():
    print(f"Độ dài: {row['transcript_length']}, Transcript: '{row['transcript']}'")

### EDA Bước 8: So sánh `transcript` từ `df_train` và `train_phones.csv`

File `train_phones.csv` thường chứa phiên âm ngữ âm (phonetic transcription) hoặc các thông tin liên quan đến cách phát âm của các từ trong `transcript` của `df_train`. Chúng ta sẽ load file này và hiển thị một vài ví dụ để xem mối quan hệ giữa chúng.

In [ ]:
import pandas as pd
import os

# Đảm bảo metadata_dir đã được định nghĩa
if 'metadata_dir' not in locals() and 'metadata_dir' not in globals():
    base_dir = ''
    metadata_dir = os.path.join(base_dir, 'metadata')

# Đường dẫn đến file train_phones.csv
train_phones_csv_path = os.path.join(metadata_dir, 'train_phones.csv')

# Đọc file train_phones.csv
df_phones = pd.read_csv(train_phones_csv_path)

print("--- Kích thước dữ liệu train_phones.csv ---")
print(df_phones.shape)

print("\n--- 5 dòng đầu tiên của df_phones ---")
display(df_phones.head())

print("\n--- So sánh mẫu Transcript và Phone --- \n(Lưu ý: Hai dataframe có thể không cùng thứ tự, đây chỉ là ví dụ để xem cấu trúc)")

# Lấy 3 mẫu ngẫu nhiên từ df_train
samples = df_train.sample(3, random_state=42) # random_state để kết quả có thể lặp lại

for index, row in samples.iterrows():
    audio_id = row['id']
    transcript = row['transcript']

    # Tìm thông tin tương ứng trong df_phones (nếu có)
    phone_info = df_phones[df_phones['id'] == audio_id]

    print(f"\nID: {audio_id}")
    print(f"  df_train Transcript: {transcript}")
    if not phone_info.empty:
        # Nếu có nhiều hơn 1 dòng phone_info cho cùng 1 ID (vd: từng từ một)
        # Ta sẽ gộp chúng lại hoặc hiển thị từng dòng
        print(f"  df_phones Phone Info:\n{phone_info.to_string(index=False)}")
    else:
        print("  df_phones Phone Info: Không tìm thấy")


## Extract Advanced Audio Features

### Subtask:
Systematically extract a comprehensive set of advanced audio features (e.g., log-Mel spectrograms, MFCCs, pitch, energy) for all audio files listed in the 'path' column of the `df_train` DataFrame. Store these features efficiently, linked by the 'id' column for later use.


In [ ]:
import pandas as pd
import os

# 1. Merge df_train and df_phones
df_merged = pd.merge(df_train[['id', 'path', 'transcript']], df_phones[['id', 'canonical']], on='id', how='inner')

# 2. Inspect the structure of df_merged
print("--- Kích thước dữ liệu df_merged ---")
print(df_merged.shape)

print("\n--- 10 dòng đầu tiên của df_merged ---")
display(df_merged.head(10))

print("\n--- Thông tin tổng quan về df_merged ---")
df_merged.info()


## 1. Import Libraries

In [ ]:
import os
import re
import json
import torch
import librosa
import numpy as np
import pandas as pd

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    Wav2Vec2FeatureExtractor,
    HubertForCTC,
    TrainingArguments,
    Trainer
)

In [ ]:
import pandas as pd
import os

# Tải lại để đảm bảo sạch dữ liệu
train_csv_path = os.path.join(metadata_dir, 'train.csv')
train_phones_csv_path = os.path.join(metadata_dir, 'train_phones.csv')

df_base = pd.read_csv(train_csv_path)
df_p = pd.read_csv(train_phones_csv_path)

# Merge: lấy 'canonical' (chuẩn) và 'transcript' (phát âm thực tế từ file phones)
# Lưu ý: Trong train_phones.csv, cột 'transcript' chứa phoneme thực tế
df_mdd = pd.merge(
    df_base[['id', 'path']],
    df_p[['id', 'canonical', 'transcript']],
    on='id',
    how='inner'
)

# Đổi tên cho rõ nghĩa
df_mdd.rename(columns={'canonical': 'target_phones', 'transcript': 'actual_phones'}, inplace=True)

display(df_mdd.head())
print(f"Kích thước tập dữ liệu MDD: {df_mdd.shape}")

## 2. Normalize Phonemes

In [ ]:
def normalize_phoneme(p_string):
    return ' '.join([
        re.sub(r'-\d+', '', p)
        for p in p_string.split()
    ])

df_mdd['target_norm'] = df_mdd['target_phones'].apply(normalize_phoneme)
df_mdd['actual_norm'] = df_mdd['actual_phones'].apply(normalize_phoneme)

print(df_mdd[
    ['target_phones', 'target_norm',
     'actual_phones', 'actual_norm']
].head())

                                       target_phones  \
0  aː-0 m aː-4 ɗ aː-2 k ɔ-4 ɓ ɛ-4 l e-0 k ɔ-4 h a...   
1  ɓ a-4 kz tʰ aː-0 ŋz ɓ ɛ-3 o-3 iz t͡ɕ i-4 nz v ...   
2  ɓ aː-0 k o-0 aː-0 iz k u-2 ŋmz ɗ ɛ-5 pz ɲ ɨ-0 ...   
3  z ɔ-1 ŋmz k ɨ-3 uz l ɔ-0 ŋmz ɗ aː-2 n ɔ-0 ɗ ə-...   
4  x i-0 ŋ i-0 n ɔ-4 iz ɗ e-4 nz s ɨ-0 t ɨ-3 t͡ɕ ...   

                                         target_norm  \
0  aː m aː ɗ aː k ɔ ɓ ɛ l e k ɔ h aː l aː k w aː ...   
1  ɓ a kz tʰ aː ŋz ɓ ɛ o iz t͡ɕ i nz v aː ŋz t͡ɕ ...   
2  ɓ aː k o aː iz k u ŋmz ɗ ɛ pz ɲ ɨ t͡ɕ a ŋz z a mz   
3  z ɔ ŋmz k ɨ uz l ɔ ŋmz ɗ aː n ɔ ɗ ə iz l aː iz...   
4  x i ŋ i n ɔ iz ɗ e nz s ɨ t ɨ t͡ɕ ɔ s u ɲ ɨ tʰ...   

                                       actual_phones  \
0  aː-0 m aː-4 ɗ aː-2 k ɔ-4 ɓ ɛ-4 l e-0 k ɔ-4 h a...   
1  ɓ a-4 kz tʰ aː-0 ŋz ɓ ɛ-3 o-3 iz t͡ɕ i-4 nz v ...   
2  ɓ aː-0 k o-0 aː-0 iz k u-2 ŋmz ɗ ɛ-5 pz ɲ ɨ-0 ...   
3  z ɔ-1 ŋmz k ɨ-3 uz l ɔ-0 ŋmz ɗ aː-2 n ɔ-0 ɗ ə-...   
4  x i-0 ŋ i-0 n ɔ-4 iz ɗ e-4 nz s ɨ-0 t ɨ-3 t

In [ ]:
df_mdd['has_error'] = (
    df_mdd['target_norm'] != df_mdd['actual_norm']
).astype(int)

print(df_mdd['has_error'].value_counts(normalize=True))

has_error
0    0.93239
1    0.06761
Name: proportion, dtype: float64


## 3. Stratified Train/Val/Test Split

In [ ]:
df_train_full, df_temp = train_test_split(
    df_mdd,
    test_size=0.2,
    random_state=42,
    stratify=df_mdd['has_error']
)

df_val_full, df_test_full = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=42,
    stratify=df_temp['has_error']
)

print(f"Train: {len(df_train_full)}")
print(f"Val:   {len(df_val_full)}")
print(f"Test:  {len(df_test_full)}")

Train: 2544
Val:   318
Test:  318


## 4. Build Vocabulary

In [ ]:
def create_vocab(df):
    all_phones = []

    for row in df['target_norm']:
        all_phones.extend(row.split())

    for row in df['actual_norm']:
        all_phones.extend(row.split())

    unique_phones = sorted(list(set(all_phones)))

    vocab = {p: i for i, p in enumerate(unique_phones)}

    vocab['[PAD]'] = len(vocab)
    vocab['[UNK]'] = len(vocab)

    return vocab

vocab = create_vocab(df_train_full)

print(f"Vocab size: {len(vocab)}")
print(list(vocab.keys())[:30])

# Save vocab
with open("vocab.json", "w") as f:
    json.dump(vocab, f, ensure_ascii=False)

Vocab size: 49
['a', 'aː', 'e', 'eaː', 'f', 'h', 'i', 'iz', 'iə', 'k', 'kpz', 'kz', 'k̟z', 'l', 'm', 'mz', 'n', 'nz', 'o', 'p', 'pz', 's', 't', 'tz', 'tʰ', 't͡ɕ', 'u', 'uz', 'uə', 'v']


In [ ]:
id2phone = {v: k for k, v in vocab.items()}

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/hubert-base-ls960"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

In [ ]:
from transformers import Wav2Vec2CTCTokenizer
from transformers import Wav2Vec2Processor

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

## 5. Dataset Class

In [ ]:
class MDDDataset(Dataset):

    def __init__(self, df, base_dir, vocab, feature_extractor):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir
        self.vocab = vocab
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        audio_path = os.path.join(
            self.base_dir,
            row['path']
        )

        speech, _ = librosa.load(audio_path, sr=16000)

        inputs = processor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        labels = [
            self.vocab.get(p, self.vocab['[UNK]'])
            for p in row['actual_norm'].split()
        ]

        return {
            "input_values": inputs.input_values.squeeze(0),
            "labels": torch.tensor(labels)
        }

In [ ]:
train_dataset = MDDDataset(
    df_train_full,
    base_dir,
    vocab,
    processor
)

val_dataset = MDDDataset(
    df_val_full,
    base_dir,
    vocab,
    processor
)

test_dataset = MDDDataset(
    df_test_full,
    base_dir,
    vocab,
    processor
)

print("Datasets ready.")

Datasets ready.


## 6. Data Collator

In [ ]:
def collate_fn(batch):

    input_values = [x["input_values"] for x in batch]
    labels = [x["labels"] for x in batch]

    input_values_padded = torch.nn.utils.rnn.pad_sequence(
        input_values,
        batch_first=True,
        padding_value=0.0
    )

    attention_mask = torch.zeros_like(
        input_values_padded,
        dtype=torch.long
    )

    for i, x in enumerate(input_values):
        attention_mask[i, :len(x)] = 1

    labels_padded = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_values": input_values_padded,
        "attention_mask": attention_mask,
        "labels": labels_padded
    }

## 7. Initialize and train model

In [ ]:
model = HubertForCTC.from_pretrained(
    "facebook/hubert-base-ls960",
    vocab_size=len(vocab),
    pad_token_id=vocab['[PAD]'],
    ctc_loss_reduction="mean"
)

model.freeze_feature_encoder()

print("Model ready.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

HubertForCTC LOAD REPORT from: facebook/hubert-base-ls960
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model ready.


In [ ]:
training_args = TrainingArguments(
    output_dir="./hubert-mdd",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-5,

    num_train_epochs=40,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=10,

    fp16=torch.cuda.is_available(),

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    remove_unused_columns=False,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    processing_class=feature_extractor
)

trainer.train()
trainer.save_model("./best_mdd_model")

Epoch,Training Loss,Validation Loss
1,16.716580,3.876301
2,16.245737,3.854439
3,15.317180,3.818392
4,15.185576,3.821230
5,15.353442,3.786220
6,15.066278,3.775015
7,15.070219,3.786590
8,14.954933,3.743255
9,14.443460,3.553248
10,13.793806,3.341364


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def ctc_decode(pred_ids, id2phone, blank_id):

    decoded = []

    previous = None

    for idx_tensor in pred_ids:
        idx = idx_tensor.item() # Convert the 0-dimensional tensor to a Python integer

        # skip blank
        if idx == blank_id:
            previous = idx
            continue

        # collapse repeats
        if idx != previous:

            token = id2phone[idx]

            # skip special tokens
            if token not in ['[PAD]', '[UNK]']:

                decoded.append(token)

        previous = idx

    return decoded

## 9. MDD Diagnosis Function

In [ ]:
def get_mdd_diagnosis(canonical, predicted):

    n, m = len(canonical), len(predicted)

    dp = np.zeros((n + 1, m + 1))

    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):

            if canonical[i-1] == predicted[j-1]:
                dp[i][j] = dp[i-1][j-1]

            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j-1],
                    dp[i-1][j],
                    dp[i][j-1]
                )

    alignment = []

    i, j = n, m

    while i > 0 or j > 0:

        if (
            i > 0 and j > 0 and
            canonical[i-1] == predicted[j-1]
        ):

            alignment.append({
                "canonical": canonical[i-1],
                "pred": predicted[j-1],
                "diagnosis": "correct"
            })

            i -= 1
            j -= 1

        elif (
            i > 0 and j > 0 and
            dp[i][j] == dp[i-1][j-1] + 1
        ):

            alignment.append({
                "canonical": canonical[i-1],
                "pred": predicted[j-1],
                "diagnosis": "substitution"
            })

            i -= 1
            j -= 1

        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:

            alignment.append({
                "canonical": canonical[i-1],
                "pred": "-",
                "diagnosis": "deletion"
            })

            i -= 1

        else:

            alignment.append({
                "canonical": "-",
                "pred": predicted[j-1],
                "diagnosis": "insertion"
            })

            j -= 1

    return alignment[::-1]

## 10. Test Samples

In [ ]:
model.eval()

sample = test_dataset[0]

with torch.no_grad():

    # Move input to the correct device and convert to model's dtype
    logits = model(
        input_values=sample["input_values"].unsqueeze(0).to(model.device).to(model.dtype)
    ).logits

pred_ids = torch.argmax(logits, dim=-1)[0]

predicted = ctc_decode(
    pred_ids,
    id2phone,
    vocab['[PAD]']
)

canonical = df_test_full.iloc[0]['target_norm'].split()

actual = df_test_full.iloc[0]['actual_norm'].split()

print("Canonical:", canonical)
print("Actual    :", actual)
print("Predicted :", predicted)

Canonical: ['ɣ', 'ə', 'nz', 't͡ɕ', 'ɨə', 'ɓ', 'aː', 'ɓ', 'aː', 'nz', 'v', 'e', 'ɲ', 'aː']
Actual    : ['ɣ', 'ə', 'nz', 't͡ɕ', 'ɨə', 'ɓ', 'aː', 'ɓ', 'aː', 'nz', 'v', 'e', 'ɲ', 'aː']
Predicted : ['h', 'əː', 'ə', 'nz', 't͡ɕ', 'ɨə', 'ɓ', 'aː', 'ɓ', 'aː', 'nz', 'v', 'e', 'ɲ', 'aː']


In [ ]:
diagnosis = get_mdd_diagnosis(
    canonical,
    predicted
)

pd.DataFrame(diagnosis)

,canonical,pred,diagnosis
0,-,h,insertion
1,ɣ,əː,substitution
2,ə,ə,correct
3,nz,nz,correct
4,t͡ɕ,t͡ɕ,correct
5,ɨə,ɨə,correct
6,ɓ,ɓ,correct
7,aː,aː,correct
8,ɓ,ɓ,correct
9,aː,aː,correct


## 11. Evaluation

In [ ]:
from sklearn.metrics import f1_score
from tqdm import tqdm
import torch
import numpy as np

# =========================
# Evaluation Loop (Đánh giá ở mức câu)
# =========================

model.eval()

total_S = 0
total_D = 0
total_I = 0
total_N = 0

y_true = []
y_pred = []

diagnosis_errors = 0
total_diagnosis = 0


for idx in tqdm(range(len(test_dataset))):

    sample = test_dataset[idx]

    input_values = sample["input_values"].unsqueeze(0).to(model.device)

    with torch.no_grad():

        logits = model(
            input_values=input_values
        ).logits

    pred_ids = torch.argmax(logits, dim=-1)[0]

    predicted = ctc_decode(
        pred_ids,
        id2phone,
        vocab['[PAD]']
    )

    canonical = df_test_full.iloc[idx]['target_norm'].split()

    actual = df_test_full.iloc[idx]['actual_norm'].split()

    diagnosis = get_mdd_diagnosis(
        canonical,
        predicted
    )

    # =========================
    # Count PER
    # =========================

    total_N += len(canonical)

    for d in diagnosis:

        if d["diagnosis"] == "substitution":
            total_S += 1

        elif d["diagnosis"] == "deletion":
            total_D += 1

        elif d["diagnosis"] == "insertion":
            total_I += 1

    # =========================
    # F1 Detection
    # =========================

    actual_has_error = (
        canonical != actual
    )

    pred_has_error = (
        canonical != predicted
    )

    y_true.append(int(actual_has_error))
    y_pred.append(int(pred_has_error))

    # =========================
    # DER
    # =========================

    gt_diag = get_mdd_diagnosis(
        canonical,
        actual
    )

    min_len = min(len(gt_diag), len(diagnosis))

    for i in range(min_len):

        total_diagnosis += 1

        if (
            gt_diag[i]["diagnosis"]
            !=
            diagnosis[i]["diagnosis"]
        ):

            diagnosis_errors += 1


# =========================
# Final Metrics
# =========================

PER = (
    total_S + total_D + total_I
) / total_N

F1 = f1_score(y_true, y_pred)

DER = (
    diagnosis_errors / total_diagnosis
    if total_diagnosis > 0 else 1.0
)

Final_Score = (
    0.5 * F1
    + 0.4 * (1 - DER)
    + 0.1 * (1 - PER)
)

print("\n--- Evaluation Results ---")

print(f"PER  : {PER:.4f}")
print(f"F1   : {F1:.4f}")
print(f"DER  : {DER:.4f}")

print(f"\nFinal MDD Score: {Final_Score:.4f}")

100%|██████████| 318/318 [00:23<00:00, 13.53it/s]


--- Evaluation Results ---
PER  : 0.2590
F1   : 0.1321
DER  : 0.2482

Final MDD Score: 0.4408


In [ ]:
from sklearn.metrics import f1_score
from tqdm import tqdm
import torch
import numpy as np
import pandas as pd
from itertools import zip_longest

# =========================
# Evaluation Loop (Đánh giá ở mức âm tiết (Phoneme Level))
# =========================

model.eval()

total_S = 0
total_D = 0
total_I = 0
total_N = 0

y_true = []
y_pred = []

diagnosis_errors = 0
total_diagnosis = 0


for idx in tqdm(range(len(test_dataset))):

    sample = test_dataset[idx]

    input_values = sample["input_values"].unsqueeze(0).to(model.device)

    with torch.no_grad():

        logits = model(
            input_values=input_values
        ).logits

    pred_ids = torch.argmax(logits, dim=-1)[0]

    predicted = ctc_decode(
        pred_ids,
        id2phone,
        vocab['[PAD]']
    )

    canonical = df_test_full.iloc[idx]['target_norm'].split()

    actual = df_test_full.iloc[idx]['actual_norm'].split()

    diagnosis = get_mdd_diagnosis(
        canonical,
        predicted
    )

    # =========================
    # Count PER
    # =========================

    total_N += len(canonical)

    for d in diagnosis:

        if d["diagnosis"] == "substitution":
            total_S += 1

        elif d["diagnosis"] == "deletion":
            total_D += 1

        elif d["diagnosis"] == "insertion":
            total_I += 1

    # =========================
    # F1 Detection
    # =========================

    gt_diag = get_mdd_diagnosis(
        canonical,
        actual
    )

    pred_diag = get_mdd_diagnosis(
        canonical,
        predicted
    )

    for gt, pred in zip_longest(
        gt_diag,
        pred_diag,
        fillvalue={"diagnosis": "missing"}
    ):

        true_error = int(
            gt["diagnosis"] != "correct"
        )

        pred_error = int(
            pred["diagnosis"] != "correct"
        )

        y_true.append(true_error)
        y_pred.append(pred_error)

    # =========================
    # DER
    # =========================

    gt_diag = get_mdd_diagnosis(
        canonical,
        actual
    )

    min_len = min(len(gt_diag), len(diagnosis))

    for i in range(min_len):

        total_diagnosis += 1

        if (
            gt_diag[i]["diagnosis"]
            !=
            diagnosis[i]["diagnosis"]
        ):

            diagnosis_errors += 1


# =========================
# Final Metrics
# =========================

PER = (
    total_S + total_D + total_I
) / total_N

F1 = f1_score(y_true, y_pred)

DER = (
    diagnosis_errors / total_diagnosis
    if total_diagnosis > 0 else 1.0
)

Final_Score = (
    0.5 * F1
    + 0.4 * (1 - DER)
    + 0.1 * (1 - PER)
)

print("\n--- Evaluation Results ---")

print(f"PER  : {PER:.4f}")
print(f"F1   : {F1:.4f}")
print(f"DER  : {DER:.4f}")

print(f"\nFinal MDD Score: {Final_Score:.4f}")

100%|██████████| 318/318 [00:11<00:00, 27.06it/s]


--- Evaluation Results ---
PER  : 0.2590
F1   : 0.0842
DER  : 0.2482

Final MDD Score: 0.4169


In [ ]:
import pandas as pd
from tqdm import tqdm
import torch

# =========================
# Generate Submission File
# =========================

model.eval()

predictions = []
test_ids = []

# Note: If there is a separate competition test set,
# replace 'test_dataset' and 'df_test_full' with that data.
for idx in tqdm(range(len(test_dataset))):

    sample = test_dataset[idx]

    # Use the real ID from the dataframe
    real_id = df_test_full.iloc[idx]['id']
    test_ids.append(real_id)

    input_values = (
        sample["input_values"]
        .unsqueeze(0)
        .to(model.device)
    )

    with torch.no_grad():
        logits = model(
            input_values=input_values
        ).logits

    # Greedy CTC decode
    pred_ids = torch.argmax(
        logits,
        dim=-1
    )[0]

    # Decoding logic with safety check for the dictionary mapping
    predicted_tokens = ctc_decode(
        pred_ids,
        id2phone,
        vocab.get('[PAD]', 47) # Default to 47 if [PAD] not found
    )

    # Convert list of phonemes to space-separated string
    pred_str = " ".join(predicted_tokens)
    predictions.append(pred_str)

# =========================
# Create predictions.csv
# =========================

submission = pd.DataFrame({
    "id": test_ids,
    "predict": predictions
})

submission.to_csv(
    "predictions.csv",
    index=False
)

print(f"\nSuccessfully saved {len(submission)} predictions to predictions.csv")

# Preview
display(submission.head())

100%|██████████| 318/318 [00:12<00:00, 25.50it/s]


Successfully saved 318 predictions to predictions.csv


,id,predict
0,GQM_Nam_6_S0006_57,h əː ə nz t͡ɕ ɨə ɓ aː ɓ aː nz v e ɲ aː
1,THA_Nu_6_S00042_13,tʰ aː ɗ i k w aː nz t iə uz k w aː s k w aː
2,NDBA_7_NAM_S0010_99,m ɛ aː uz tʰ ə iz k ɔ m o s o n əː iz l aː ŋz ...
3,NMA_7_NU_S0002_20,s ɛ mz v ɔ iz l aː h ɛ h e tz
4,LXT_Nam_6_S0008_38,m o tz nz h o iz l o v aː m eaː iz t͡ɕ o iz s ...


### So sánh kết quả dự đoán với Ground Truth (train_phones.csv)
Chúng ta sẽ lấy các ID trong tập test để so sánh chuỗi phoneme dự đoán và chuỗi phoneme chuẩn (canonical) từ dữ liệu gốc.

In [ ]:
import pandas as pd

# Kết hợp kết quả dự đoán (submission) với thông tin gốc (df_test_full)
# Lưu ý: df_test_full đã chứa cột 'target_norm' (phoneme chuẩn đã normalize)
comparison_df = pd.merge(
    submission[['id', 'predict']],
    df_test_full[['id', 'target_norm']],
    on='id'
)

# Đổi tên cột để dễ quan sát
comparison_df.columns = ['ID', 'Dự đoán (Model)', 'Chuẩn (Ground Truth)']

# Hiển thị 20 mẫu ngẫu nhiên để so sánh
print("--- So sánh ngẫu nhiên 20 mẫu trong tập Test ---")
display(comparison_df.sample(min(20, len(comparison_df)), random_state=42))

# Tính toán tỷ lệ khớp hoàn toàn ở mức chuỗi
exact_matches = (comparison_df['Dự đoán (Model)'] == comparison_df['Chuẩn (Ground Truth)']).sum()
print(f"\nSố lượng mẫu khớp hoàn toàn: {exact_matches} / {len(comparison_df)} ({exact_matches/len(comparison_df)*100:.2f}%)")

--- So sánh ngẫu nhiên 20 mẫu trong tập Test ---


,ID,Dự đoán (Model),Chuẩn (Ground Truth)
73,CNBA_Nu_6_S0003_20,k o ŋmz t͡ɕ w ɲ ə nz z aː i ŋ̟z iə ŋz eaː ŋ̟z,k o ŋmz t͡ɕ uə ɲ ə nz z aː t iə ŋz e k̟z
277,NMA_7_NU_S0002_221,k ɛ nz ɲ iə nz ɓ iə nz z ə iz m iə nz l u ŋmz ...,k ɛ m iə nz ɓ iə nz ŋ ɨə iz m iə nz n w i x i ...
25,DNK_7_NU_S0048_276,t o iz ɲ iə tz k ɔ nz t͡ɕ i mz h i nz n ɔ k ɔ ...,t o iz ŋ iə pz k ɔ nz t͡ɕ i mz x i n ɔ k ɔ nz ...
256,NTN_7_NU_S0042_92,əː mz t͡ɕ aː ŋz v aː tʰ ŋz h ɔ,ə mz t͡ɕ aː ŋz v əː tʰ əː mz tʰ ɔ
9,NQH_7_NAM_S0049_29,t i mz ɲ ɨ ŋz t͡ɕ i t͡ɕ iə tz t͡ɕ ɔ tʰ ə iz h ...,t i mz ɲ ɨ ŋz t͡ɕ i t iə tz t͡ɕ ɔ tʰ ə iz h aː...
101,VTA_Nam_6_S0002_14,h ɨə ŋz t͡ɕ ɨə nz ɲ aː ŋz h aː tz v ɨə tz t͡ɕ ...,h o mz t͡ɕ ɨə kz ɛ mz x w aː kz l aː kz t͡ɕ i ...
176,CNBA_Nu_6_S0003_164,v i k aː m ə tz h aː l ɔ,v i k aː m ə pz h aː iz n ɔ
186,PTL_7_NU_S0008_78,tʰ iə nz n aː ɲ ɛ k ɔ ŋmz k ɔ nz t͡ɕ e nz l ɨ ...,tʰ iə nz ŋ aː m ɛ k ɔ ŋmz k ɔ nz t͡ɕ e nz l ɨ ...
63,NNM_Nu_6_S00045_81,s əː iz x w iz ɓ a iz h aː iz v əː nz h eaː ŋ̟z,s ɔ iz tʰ o iz ɓ a iz ɲ aː l əː nz eaː ŋ̟z
116,PC_7_NAM_S0004_84,ɗ aː n aː ŋz k ɔ nz k ɔ k aː k ɔ z o ŋmz,əː ɗ aː n a ŋz k ɔ nz k ɔ k aː k ə uz z o ŋmz



Số lượng mẫu khớp hoàn toàn: 21 / 318 (6.60%)
